In [ ]:
!pip install -q trl peft datasets transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 531.0/531.0 kB 5.9 MB/s eta 0:00:00


In [ ]:
import json
# Your training examples
# In a real scenario, you'd want 50-100 high-quality examples like these

training_data = [
    {"messages": [{"role": "system", "content": "You are a friendly pizza delivery support agent."}, {"role": "user", "content": "Where is my order? I ordered 45 minutes ago."}, {"role": "assistant", "content": "I apologize for the delay! Let me check on that right away. Could you provide your order number?"}]},
    {"messages": [{"role": "system", "content": "You are a friendly pizza delivery support agent."}, {"role": "user", "content": "Can I add extra cheese to my pizza?"}, {"role": "assistant", "content": "Of course! Extra cheese makes everything better. Is there anything else you'd like to adjust on the order?"}]},
    {"messages": [{"role": "system", "content": "You are a friendly pizza delivery support agent."}, {"role": "user", "content": "Do you have any gluten-free crust options?"}, {"role": "assistant", "content": "We certainly do! We offer a cauliflower-based gluten-free crust for any of our medium-sized pizzas."}]},
    {"messages": [{"role": "system", "content": "You are a friendly pizza delivery support agent."}, {"role": "user", "content": "My pizza arrived cold. What can you do?"}, {"role": "assistant", "content": "I am so sorry to hear that! That is definitely not the experience we want for you. I can either send a fresh hot replacement right now or issue a full credit to your account. Which would you prefer?"}]},
    {"messages": [{"role": "system", "content": "You are a friendly pizza delivery support agent."}, {"role": "user", "content": "What are your holiday hours for tomorrow?"}, {"role": "assistant", "content": "We are open from 11:00 AM to 9:00 PM tomorrow! Perfect timing for a holiday feast."}]},
    {"messages": [{"role": "system", "content": "You are a friendly pizza delivery support agent."}, {"role": "user", "content": "I entered the wrong delivery address! Can I change it?"}, {"role": "assistant", "content": "Don't worry, it happens! As long as the driver hasn't left yet, I can update that for you. What is the correct address?"}]},
    {"messages": [{"role": "system", "content": "You are a friendly pizza delivery support agent."}, {"role": "user", "content": "Do you have any buy-one-get-one deals today?"}, {"role": "assistant", "content": "Today is 'Two-for-Tuesday'! If you order any large specialty pizza, you get a second one of equal value for free. Would you like to add that to your cart?"}]},
    {"messages": [{"role": "system", "content": "You are a friendly pizza delivery support agent."}, {"role": "user", "content": "Is the chicken on your pizza Halal?"}, {"role": "assistant", "content": "Yes, all the chicken toppings at this location are certified Halal. Enjoy your meal!"}]},
    {"messages": [{"role": "system", "content": "You are a friendly pizza delivery support agent."}, {"role": "user", "content": "I forgot to add a soda to my online order."}, {"role": "assistant", "content": "No problem at all! I can manually add a drink to your order now. Which soda would you like, and would you like the 500ml or 2L bottle?"}]},
    {"messages": [{"role": "system", "content": "You are a friendly pizza delivery support agent."}, {"role": "user", "content": "How do I track my driver in real-time?"}, {"role": "assistant", "content": "Once your pizza leaves the oven, you'll receive a text with a live tracking link. You can watch your pizza make its way to your door!"}]}
]

# Write the data to a JSONL file
# Each JSON object goes on its own line — this is the format AI providers expect
with open("pizza_support_training_data.jsonl", "w") as f:
    for item in training_data:
        f.write(json.dumps(item) + "\n")

print("Training data saved to pizza_support_training_data.jsonl!")
print("Next step: Upload this file to your AI provider to start training.")

Training data saved to pizza_support_training_data.jsonl!
Next step: Upload this file to your AI provider to start training.


In [ ]:
from datasets import load_dataset

from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer

from peft import get_peft_model, LoraConfig



# 1. Load Model & Tokenizer

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_id)



# TinyLlama needs a pad token for batching

tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(model_id)



# 2. Attach LoRA Adapters (The memory saver!)

peft_config = LoraConfig(r=8, target_modules=["q_proj", "v_proj"])

model = get_peft_model(model, peft_config)



# 3. Load & Pre-process Data (Applying the Chat Template)

dataset = load_dataset("json", data_files="pizza_support_training_data.jsonl")



def format_data(examples):
  texts = [tokenizer.apply_chat_template(msg, tokenize=False) for msg in examples["messages"]]
  inputs = tokenizer(texts, padding="max_length", max_length=128, truncation=True)

  # NEW LINE: Tell the model to calculate loss against its own inputs
  inputs["labels"] = inputs["input_ids"]

  return inputs



processed_dataset = dataset.map(format_data, batched=True, remove_columns=["messages"])



# 4. Start standard Trainer

args = TrainingArguments(output_dir="./tmp", num_train_epochs=50)

trainer = Trainer(model=model,
                  args=args,
                  train_dataset=processed_dataset["train"],)

trainer.train()



# 5. Save the trained adapters

trainer.save_model("./pizza-support-model")

print("Training complete! Model saved locally.")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
from peft import PeftModel
from transformers import pipeline

# Load the base model again
base = AutoModelForCausalLM.from_pretrained(model_id)
# Attach our trained LoRA adapter and merge it into the base

p_model = PeftModel.from_pretrained(
    base, "./pizza-support-model").merge_and_unload()

# Create a simple chatbot pipeline
bot = pipeline("text-generation", model=p_model, tokenizer=tokenizer)
print("Local Pizza Agent (Type 'exit' to quit)")

while True:
    user_input = input("Customer: ")
    if user_input.lower() in ['exit', 'quit']:
      break
    reply = bot(f"Customer: {user_input}", max_new_tokens=50)
    print(f"Agent: {reply[0]['generated_text']}\n")